# Patch grid geometry and padding

This notebook confirms how `Patcher.build` lays a regular grid of overlapping patches over a
spatial extent, how it computes symmetric padding so the grid covers the whole image, and how
many patches result. We overlay the patch rectangles on the canvas and inspect the resulting
`GridInfo` fields.

Modules exercised:

- `pipelines.dataset_pipeline.spatial.Patcher`
- `pipelines.dataset_pipeline.spatial.GridInfo`

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path("../..").resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import numpy             as np
import matplotlib.pyplot as plt

import matplotlib
matplotlib.rcParams["figure.dpi"] = 110
matplotlib.rcParams["image.interpolation"] = "nearest"

SEED = 0
np.random.seed(SEED)

try:
    import torch
    torch.manual_seed(SEED)
except Exception:
    pass

print("repo root on path:", REPO_ROOT)



## Build a grid

We choose a spatial size that is not an integer multiple of the patch stride, so that padding
is required. The `GridInfo` returned by the patcher records the grid shape and padding.


In [ ]:
from pipelines.dataset_pipeline.spatial import Patcher

H, W       = 70, 90
patch_size = (32, 32)
stride     = 24

patcher = Patcher.build(spatial_size=(H, W), patch_size=patch_size, stride=stride)
grid    = patcher.grid

for k, v in grid.as_dict().items():
    print(f"{k:24s}: {v}")



## Patch placement on the canvas

Each rectangle below is the in-bounds region of one patch (before per-patch padding). The grid
of `n_v` by `n_h` patches should tile the full image with the configured overlap.


In [ ]:
import matplotlib.patches as mpatches

fig, ax = plt.subplots(figsize=(6, 5))
ax.imshow(np.zeros((H, W)), cmap="gray", vmin=0, vmax=1)

colors = plt.cm.tab20(np.linspace(0, 1, grid.number_of_patches))
for idx, (v0c, v1c, h0c, h1c, _pw) in enumerate(patcher._patch_coords):
    rect = mpatches.Rectangle((h0c, v0c), h1c - h0c, v1c - v0c, fill=False, edgecolor=colors[idx], linewidth=1.5)
    ax.add_patch(rect)

ax.set_title(f"{grid.n_v} x {grid.n_h} = {grid.number_of_patches} patches over {H} x {W}")
ax.set_xlabel("range"); ax.set_ylabel("azimuth")
plt.tight_layout()
plt.show()



## Padding required to cover the image

The patcher pads the canvas symmetrically so the last patch reaches the border. We visualise
the padded extent against the original image rectangle.


In [ ]:
ph, pw = grid.padded_size
fig, ax = plt.subplots(figsize=(6, 5))
ax.add_patch(mpatches.Rectangle((-grid.pad_left, -grid.pad_top), pw, ph, fill=True, facecolor="0.85", edgecolor="0.4", label="padded extent"))
ax.add_patch(mpatches.Rectangle((0, 0), W, H, fill=False, edgecolor="C3", linewidth=2.0, label="original image"))

ax.set_xlim(-grid.pad_left - 5, W + grid.pad_right + 5)
ax.set_ylim(H + grid.pad_bot + 5, -grid.pad_top - 5)
ax.set_title(f"padding  top={grid.pad_top} bot={grid.pad_bot} left={grid.pad_left} right={grid.pad_right}")
ax.legend(loc="upper right")
ax.set_xlabel("range"); ax.set_ylabel("azimuth")
plt.tight_layout()
plt.show()



## Sweep over strides

Reducing the stride increases overlap and therefore the patch count. We confirm the trend
predicted by the grid formula.


In [ ]:
strides = [8, 16, 24, 32]
counts  = []
for s in strides:
    g = Patcher.build(spatial_size=(H, W), patch_size=patch_size, stride=s).grid
    counts.append(g.number_of_patches)
    print(f"stride {s:3d}: n_v={g.n_v} n_h={g.n_h} patches={g.number_of_patches}")

fig, ax = plt.subplots(figsize=(5, 3))
ax.plot(strides, counts, "o-")
ax.set_xlabel("stride [pixels]"); ax.set_ylabel("number of patches")
ax.set_title("patch count vs stride")
plt.tight_layout()
plt.show()


## Expected visual outcome

The coloured rectangles should tile the entire canvas with visible overlap, numbering exactly
`n_v * n_h`. The padded extent should be a grey rectangle slightly larger than and symmetric
about the original red image rectangle. The stride sweep should show patch count decreasing
monotonically as stride grows.